# Etapa 5i — Relleno de exudados antes de los filtros

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** En retinopatía, entre el 5 % y el 34 % de los falsos positivos están lejos de los vasos, sobre lesiones (`05h`). Las lesiones claras (exudados) tienen bordes que los filtros toman por vasos: la franja oscura entre dos exudados parece un vaso. Annunziata et al. (2016) rellenan los exudados con una estimación de su entorno **antes** del realce, y reportan en esta misma base una exactitud de 95,81 %.

**Candidatas** (sobre el algoritmo actual: normalización por división, 28 mapas, entrenamiento equilibrado por grosor, valor fijo elegido en entrenamiento, 5 grupos de personas):
- **A.** Sin relleno (el algoritmo actual).
- **B.** Con relleno de exudados (`preprocess.rellenar_exudados`): es candidato lo que supera en 4 desvíos robustos el brillo relativo del verde (verde / fondo − 1), fuera de un círculo de radio 10 % del diámetro alrededor del disco óptico y con un área mínima; se dilata solo hacia píxeles más claros que el fondo (nunca tapa un vaso) y se rellena con la media gaussiana de los píxeles de fondo cercanos. Parámetros de criterio, en proporción al diámetro; no se ajustaron con datos. En desarrollo rellena entre 0,1 % y 2 % del campo, y casi nada que sea vaso según los expertos (de 0 % a 4 % de lo rellenado).

**Medidas**, en las personas de afuera, a resolución original, en el campo de visión de la base: Matthews, Dice de línea central, vasos finos (sensibilidad de línea central) y precisión. Descriptivo: por grupo, con atención a retinopatía.

**Regla de adopción (fijada antes de correr).** B reemplaza a A si
1. mejora Matthews con un intervalo al 95 % que no incluye el cero, **y**
2. no es inferior a A en vasos finos ni en Dice de línea central (límite inferior > −0,03).

Si no cumple, queda A.

**Control:** A tiene que reproducir el Matthews de B en `05h` (0,7520).

In [1]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.preprocess import SUFIJO_RELLENO
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
CANDS = {"A. sin relleno": C.SEGMENT_BACKGROUND_MODE, "B. con relleno": C.SEGMENT_BACKGROUND_MODE + SUFIJO_RELLENO}
ORDER = list(CANDS)
WORKERS = 4
print(CANDS, "|", len(C.SEGMENT_FEATURES), "mapas")

{'A. sin relleno': 'división', 'B. con relleno': 'división + relleno de exudados'} | 28 mapas


## Pasada 1: píxeles de entrenamiento y de elección del umbral

In [2]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, {c: training_pixels(image_id, background=m) for c, m in CANDS.items()}

t = time.time()
P1 = dict(map_images(pass1, ids, workers=WORKERS))
samples = {c: {i: P1[i][c] for i in ids} for c in CANDS}
del P1
print(f"Pasada 1: {time.time() - t:.0f} s")

Pasada 1: 446 s


## Ajuste por vuelta

In [3]:
fit = {}
for k in range(5):
    tr = list(folds.index[folds != k])
    for c in CANDS:
        w = fit_weights(samples[c], tr)
        fit[(k, c)] = (w, choose_value(samples[c], w, tr))
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)]).round(4)

,A. sin relleno,B. con relleno
vuelta 0,0.8784,0.8614
vuelta 1,0.8751,0.8735
vuelta 2,0.8609,0.8764
vuelta 3,0.8626,0.8619
vuelta 4,0.8712,0.8708


## Pasada 2: máscaras de las personas de afuera, a resolución original

In [4]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    rows = []
    for c, mode in CANDS.items():
        w, value = fit[(k, c)]
        (p,), _, _ = feature_scores(rgb, [w], 2, background=mode)
        m = (upscale(p, fov.shape) > value) & auto
        marked = m & fov
        rows.append({"image": image_id, "candidata": c, "matthews": matthews(m, truth, fov),
                     "dice_linea_central": centerline_dice(m, truth, fov), "finos": m[thin].mean(),
                     "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05i_exudados.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

Pasada 2: 411 s


## Resultados

In [5]:
out = []
for meas in ("matthews", "dice_linea_central", "finos", "precision"):
    w = R.pivot(index="image", columns="candidata", values=meas)[ORDER]
    d = (w["B. con relleno"] - w["A. sin relleno"]).to_frame()
    lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
    out.append({"medida": meas, "A": w["A. sin relleno"].mean(), "B": w["B. con relleno"].mean(),
                "B − A": d.iloc[:, 0].mean(), "inferior": lo[0], "superior": hi[0],
                "imágenes que mejoran": int((d.iloc[:, 0] > 0).sum())})
table = pd.DataFrame(out).set_index("medida")
table.round(4)

,A,B,B − A,inferior,superior,imágenes que mejoran
medida,,,,,,
matthews,0.7520,0.7533,0.0013,0.0000,0.0030,19
dice_linea_central,0.7563,0.7583,0.0020,0.0006,0.0036,22
finos,0.3514,0.3534,0.0020,-0.0049,0.0088,21
precision,0.7987,0.8002,0.0016,-0.0025,0.0059,14


In [6]:
t = table
ok = bool(t.loc["matthews", "inferior"] > 0 and t.loc["finos", "inferior"] > -C.NONINF_MARGIN_MATTHEWS
          and t.loc["dice_linea_central", "inferior"] > -C.NONINF_MARGIN_MATTHEWS)
print("B cumple la regla:", ok, "→ se adopta:", "B. con relleno" if ok else "A. sin relleno")

B cumple la regla: True → se adopta: B. con relleno


In [7]:
print("Por grupo (media):")
R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group",
                                              values=["matthews", "precision", "finos"]).loc[ORDER].round(3)

Por grupo (media):


finos               matthews               precision              
group              dr      g      h       dr      g      h        dr      g      h
candidata                                                                         
A. sin relleno  0.370  0.339  0.345    0.704  0.755  0.797     0.708  0.771  0.918
B. con relleno  0.371  0.340  0.350    0.707  0.755  0.798     0.712  0.771  0.918

## Decisión (escrita después de correr)

**Control:** A reproduce exacto el Matthews de `05h` (0,7520).

**Se adopta B, el relleno de exudados** (`config.SEGMENT_BACKGROUND_MODE = "división + relleno de exudados"`), porque cumple la regla, **en el límite y con un efecto mínimo:**

| B contra A (personas de afuera) | Diferencia (intervalo 95 %) | Imágenes que mejoran |
|---|---|---|
| Matthews | +0,0013 (+0,00005 a +0,0030) | 19 de 30 (11 de 18 personas) |
| Dice de línea central | +0,0020 (+0,0006 a +0,0036) | 22 de 30 |
| Vasos finos | +0,2 puntos (−0,5 a +0,9) | 21 de 30 |
| Precisión | +0,2 puntos (−0,3 a +0,6) | 14 de 30 |

- **La ganancia está donde se esperaba:** retinopatía +0,0026 (Matthews 0,704 → 0,707; precisión 0,708 → 0,712), glaucoma y sanos +0,0007. La imagen que más gana es 06_dr (+0,020); la que más pierde, 13_dr (−0,003).
- **Es chico porque las lesiones sueltas son la minoría de los falsos positivos de retinopatía** (entre 5 % y 34 %; `05h`): el resto es diferencia de ancho con lo marcado, que el relleno no toca.

**Cuidados.** El límite inferior apenas excluye el cero, y es la tercera ronda que se decide con las mismas 30 imágenes: mejoras de este tamaño quedan dentro del optimismo que agrega elegir varias veces con los mismos datos. Se adopta por la regla fijada, sin desvío de criterio, y se informa como una mejora mínima. El conjunto de prueba no se usó.